# 16 · Capstone Project — Retail Analytics

Time to be the hero. 🦸 You're the new data analyst at our retail company. The
leadership team has questions; answer them with SQL. Each task lists the skills
it exercises. Try each yourself, then reveal the solution.

Run the setup cell, then work through the challenges.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

### Challenge 1 — Monthly revenue trend
*Skills: joins, aggregation, dates.*

Report total revenue per month (only `completed` orders), ordered by month.

**✏️ Exercise 1.** Compute completed-order revenue for each `YYYY-MM`.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT STRFTIME('%Y-%m', o.order_date) AS month,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY month
ORDER BY month;

### Challenge 2 — Top customers
*Skills: joins, aggregation, ranking.*

Rank customers by lifetime revenue (completed orders). Show name, revenue, and rank; return the top 5.

**✏️ Exercise 2.** Rank customers by total revenue and show the top 5.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH rev AS (
  SELECT o.customer_id, SUM(oi.quantity * oi.unit_price) AS revenue
  FROM orders o
  JOIN order_items oi ON o.order_id = oi.order_id
  WHERE o.status = 'completed'
  GROUP BY o.customer_id
)
SELECT cu.first_name || ' ' || cu.last_name AS customer,
       ROUND(rev.revenue, 2) AS revenue,
       RANK() OVER (ORDER BY rev.revenue DESC) AS rnk
FROM rev JOIN customers cu ON rev.customer_id = cu.customer_id
ORDER BY rnk
LIMIT 5;

### Challenge 3 — Best-selling product per category
*Skills: joins, window functions, CTE.*

For each category, find the product with the highest total units sold.

**✏️ Exercise 3.** Top product (by units sold) within each category.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH sales AS (
  SELECT p.category_id, p.product_name, SUM(oi.quantity) AS units
  FROM order_items oi
  JOIN products p ON oi.product_id = p.product_id
  GROUP BY p.category_id, p.product_name
),
ranked AS (
  SELECT category_id, product_name, units,
         ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY units DESC) AS rn
  FROM sales
)
SELECT c.category_name, r.product_name, r.units
FROM ranked r
JOIN categories c ON r.category_id = c.category_id
WHERE r.rn = 1
ORDER BY c.category_name;

### Challenge 4 — Customer segmentation
*Skills: CASE, aggregation, joins.*

Label each customer 'VIP' (revenue >= 300), 'Regular' (>= 100), 'New/Low' (> 0), or 'No orders' (0).

**✏️ Exercise 4.** Segment every customer by lifetime completed revenue.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH rev AS (
  SELECT cu.customer_id,
         COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS revenue
  FROM customers cu
  LEFT JOIN orders o      ON cu.customer_id = o.customer_id AND o.status = 'completed'
  LEFT JOIN order_items oi ON o.order_id = oi.order_id
  GROUP BY cu.customer_id
)
SELECT cu.first_name || ' ' || cu.last_name AS customer,
       ROUND(rev.revenue, 2) AS revenue,
       CASE WHEN rev.revenue >= 300 THEN 'VIP'
            WHEN rev.revenue >= 100 THEN 'Regular'
            WHEN rev.revenue > 0    THEN 'New/Low'
            ELSE 'No orders' END AS segment
FROM rev JOIN customers cu ON rev.customer_id = cu.customer_id
ORDER BY revenue DESC;

### Challenge 5 — Sales rep leaderboard
*Skills: joins, aggregation, self-join.*

For each sales rep, show their manager and the total revenue they generated from completed orders.

**✏️ Exercise 5.** Revenue per employee, with their manager's name.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT e.first_name || ' ' || e.last_name AS employee,
       m.first_name || ' ' || m.last_name AS manager,
       ROUND(COALESCE(SUM(oi.quantity * oi.unit_price), 0), 2) AS revenue
FROM employees e
LEFT JOIN employees m   ON e.manager_id = m.employee_id
LEFT JOIN orders o      ON o.employee_id = e.employee_id AND o.status = 'completed'
LEFT JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY e.employee_id
ORDER BY revenue DESC;

## 🎉 Congratulations — you finished the SQL Zero-to-Hero Bootcamp!

You can now:
- query, filter, sort, and shape data
- aggregate and group
- join many tables and reason about relationships
- write subqueries, CTEs, and window functions
- define schema (DDL), modify data (DML), and use transactions safely
- build views and indexes

**Where to go next:** try these queries against your own data, learn your
target database's dialect (PostgreSQL, MySQL, SQL Server), and practice on
real datasets. Keep this project around as a reference — re-run
`build_notebooks.py` any time to reset the exercises.